In [25]:
import numpy as np
import pandas as pd

import geopandas as gpd
from shapely.geometry import Point

import matplotlib as plt
import folium

**Import National Park Data**

I'm starting by importing national park data from a google sheet I created. I copied the data over from a public dataset

In [38]:
sheet_id = "1cwnOCRJdpOT0i8lIlSPEtDVLwUdPPsI0L33EWKOGcwk"
url = f"https://docs.google.com/spreadsheets/d/1cwnOCRJdpOT0i8lIlSPEtDVLwUdPPsI0L33EWKOGcwk/gviz/tq?tqx=out:csv"
df = pd.read_csv(url)

print(df.head())

    National Park  Longitude  Latitude        State(s)   Park Established  \
0          Acadia     -68.21     44.35           Maine  February 26, 1919   
1  American Samoa    -170.68    -14.25  American Samoa   October 31, 1988   
2          Arches    -109.57     38.68            Utah  November 12, 1971   
3        Badlands    -102.50     43.75    South Dakota  November 10, 1978   
4        Big Bend    -103.25     29.25           Texas      June 12, 1944   

                     Area of Park Number of Park Visitors (2022)  Visited  
0     49,075.26 acres (198.6 km2)                      3,970,260        1  
1       8,256.67 acres (33.4 km2)                         12,135        0  
2     76,678.98 acres (310.3 km2)                      1,460,652        0  
3    242,755.94 acres (982.4 km2)                      1,006,809        1  
4  801,163.21 acres (3,242.2 km2)                        514,107        0  


**Convert to geopandas data frame**

To be able to plot the park data on a folium map, I'll need to convert the pandas dataframe to a geopandas dataframe. First, I need to make a geometry column. I can then convert to gpd and update the CRS.

In [39]:
# Create geometry column from lon/lat
geometry = [Point(xy) for xy in zip(df["Longitude"], df["Latitude"])]

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry=geometry)

# Set the coordinate reference system (CRS) to WGS84 (lat/lon)
gdf.set_crs(epsg=4326, inplace=True)

print(gdf.head())

    National Park  Longitude  Latitude        State(s)   Park Established  \
0          Acadia     -68.21     44.35           Maine  February 26, 1919   
1  American Samoa    -170.68    -14.25  American Samoa   October 31, 1988   
2          Arches    -109.57     38.68            Utah  November 12, 1971   
3        Badlands    -102.50     43.75    South Dakota  November 10, 1978   
4        Big Bend    -103.25     29.25           Texas      June 12, 1944   

                     Area of Park Number of Park Visitors (2022)  Visited  \
0     49,075.26 acres (198.6 km2)                      3,970,260        1   
1       8,256.67 acres (33.4 km2)                         12,135        0   
2     76,678.98 acres (310.3 km2)                      1,460,652        0   
3    242,755.94 acres (982.4 km2)                      1,006,809        1   
4  801,163.21 acres (3,242.2 km2)                        514,107        0   

                 geometry  
0    POINT (-68.21 44.35)  
1  POINT (-170.68 

**Simple Plot Test**

Eventually, I want to make some fun maps... but I'll start with a POC of simple blue points for parks I've visited and black points for parks I haven't

In [43]:

# Pick a central location for the map (US center-ish)
m = folium.Map(location=[39.8283, -98.5795], zoom_start=4)

# Define color mapping
color_map = {0: "black", 1: "blue"}

# Add points to the map
for _, row in gdf.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=4,
        color=color_map[row["Visited"]],
        fill=True,
        fill_opacity=0.8,
        popup=row.get("National Park", None)  # shows name if column exists
    ).add_to(m)

# Show map
m.save("visited_map.html")